# Learned Stage-Wise Curvature Tuning on DermaMNIST

This notebook evaluates Learned Stage-Wise Curvature Tuning (SW-CT) on the DermaMNIST dataset using an ImageNet-pretrained ResNet-50.

DermaMNIST is a medical image-classification dataset containing dermatological lesion images. Its visual domain differs from the natural images used for ImageNet pretraining, making it useful for testing whether Learned SW-CT remains effective under substantial domain shift.

The experiments compare four conditions:

1. **Baseline** — the original pretrained ResNet with ReLU activations and a trained downstream classifier.
2. **Single-Parameter Curvature Tuning (S-CT)** — one global beta selected through validation-based exhaustive search.
3. **Learned SW-CT with fixed initialization** — four trainable stage-wise beta parameters initialized at 0.78.
4. **Learned SW-CT with random initialization** — four trainable stage-wise beta parameters independently initialized between 0.70 and 0.99.

Each condition is evaluated using seeds 42, 43, and 44.

Experiments are conducted for both 20 and 30 epochs to determine whether additional optimization improves performance or changes the learned stage-wise beta values.

Wall-clock execution time is also recorded to compare the computational cost of Baseline, S-CT, and Learned SW-CT.

In [1]:
!nvidia-smi

Sun Oct  4 11:45:59 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.178.04             Driver Version: 580.178.04     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   47C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
%cd /content

!git clone -b stage-wise-ct \
    https://github.com/Syed1611/curvature-tuning-research.git

%cd /content/curvature-tuning-research/src/curvature-tuning

!git branch --show-current
!git log -1 --oneline

/content
Cloning into 'curvature-tuning-research'...
remote: Enumerating objects: 575, done.
remote: Counting objects: 100% (247/247), done.
remote: Compressing objects: 100% (123/123), done.
remote: Total 575 (delta 153), reused 181 (delta 103), pack-reused 328 (from 1)
Receiving objects: 100% (575/575), 6.93 MiB | 5.64 MiB/s, done.
Resolving deltas: 100% (284/284), done.
/content/curvature-tuning-research/src/curvature-tuning
stage-wise-ct
068c466 (HEAD -> stage-wise-ct, origin/stage-wise-ct, origin/HEAD) Updated aircraft with complete results


## Environment Setup

The repository dependencies are installed before running the experiments.

After installation, the Colab runtime should be restarted so that the required package versions are loaded correctly.

In [3]:
%pip install -q -r requirements.txt
%pip install --force-reinstall --no-cache-dir "numpy==1.26.3"

  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.7/15.7 MB 82.9 MB/s eta 0:00:0000:010:01
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.4/42.4 kB 2.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.6/57.6 kB 3.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.9/89.9 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.9/64.9 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 133.2/133.2 kB 8.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 81.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.4/78.4 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.7/12.7 MB 106.9 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.3/8

In [1]:
%cd /content/curvature-tuning-research/src/curvature-tuning

import os
import torch

os.environ["WANDB_MODE"] = "disabled"

print("Working directory:", os.getcwd())
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

/content/curvature-tuning-research/src/curvature-tuning
Working directory: /content/curvature-tuning-research/src/curvature-tuning
PyTorch: 2.11.0+cu128
CUDA available: True
GPU: Tesla T4


## DermaMNIST Dataset

Before running the experiments, the DermaMNIST data loaders are checked to confirm that the downstream dataset can be loaded correctly.

The same dataset split and random seed protocol are used for Baseline, S-CT, and Learned SW-CT so that the methods are compared under the same conditions.

In [2]:
from utils.data import get_data_loaders

train_loader, test_loader, val_loader = get_data_loaders(
    "imagenet_to_medmnist/dermamnist",
    train_batch_size=32,
    test_batch_size=128,
    seed=42,
)

print("Train samples:", len(train_loader.dataset))
print("Validation samples:", len(val_loader.dataset))
print("Test samples:", len(test_loader.dataset))

images, labels = next(iter(train_loader))

print("Image batch shape:", images.shape)
print("Label batch shape:", labels.shape)
print("Labels:", labels[:10])

README.md:   0%|          | 0.00/7.32k [00:00<?, ?B/s]

medmnist.py:   0%|          | 0.00/4.84k [00:00<?, ?B/s]

Generating train split: 0 examples [00:00, ? examples/s]

Generating test split: 0 examples [00:00, ? examples/s]

Generating validation split: 0 examples [00:00, ? examples/s]

Train samples: 7007
Validation samples: 1003
Test samples: 2005
Image batch shape: torch.Size([32, 3, 224, 224])
Label batch shape: torch.Size([32])
Labels: tensor([4, 5, 2, 5, 5, 3, 5, 5, 5, 5])


# ResNet-50 Experiments

ResNet-50 is evaluated first because it was used during the initial development of Learned SW-CT.

The experiment compares Baseline, S-CT, fixed-initialization SW-CT, and random-initialization SW-CT.

All methods use the same ImageNet-pretrained backbone, DermaMNIST dataset split, experimental seeds, and downstream training protocol.

## ResNet-50 — 20-Epoch Evaluation

The first evaluation uses 20 training epochs.

This provides the primary reference experiment and maintains compatibility with the earlier Learned SW-CT work.

### Methods

**Baseline**

The pretrained ResNet-50 retains its original ReLU activations. The backbone is frozen and only the downstream classifier is trained.

**S-CT**

The original Single-Parameter Curvature Tuning method searches over a single beta shared throughout the network. Candidate beta values from 0.70 to 1.00 are evaluated using validation performance, and the best candidate is evaluated on the test set.

The recorded S-CT execution time includes the complete beta search rather than only the final selected beta.

**Learned SW-CT — Fixed Initialization**

Four trainable beta parameters are assigned to the four ResNet stages. All four begin at beta = 0.78.

This preserves the initialization used in the original Learned SW-CT experiments.

**Learned SW-CT — Random Initialization**

Four trainable stage-wise beta parameters are initialized independently between 0.70 and 0.99.

The random initialization is generated deterministically from the experiment seed. This allows the experiment to test whether Learned SW-CT depends on the fixed 0.78 starting value while remaining reproducible.

In [ ]:
!CUDA_VISIBLE_DEVICES=1 WANDB_MODE=disabled python generalization_ct.py \
    --model resnet50 \
    --pretrained_ds imagenet \
    --transfer_ds medmnist/dermamnist \
    --seed 42 \
    --linear_probe_train_bs 32 \
    --linear_probe_test_bs 128 \
    --epochs 20

2026-10-03 14:21:06.378 | INFO     | __main__:main:74 - Log file: ./logs/generalization_ct_imagenet_to_medmnist-dermamnist_resnet50_seed42_epochs20.log
2026-10-03 14:21:06.380 | INFO     | __main__:main:78 - Running on cuda
2026-10-03 14:21:09.284 | INFO     | __main__:main:105 - Testing baseline...
2026-10-03 14:21:09.360 | INFO     | __main__:main:116 - Number of trainable parameters: 14343
2026-10-03 14:21:09.360 | INFO     | __main__:main:117 - Starting transfer learning...
2026-10-03 14:21:36.235 | INFO     | train:train_epoch:47 - Epoch 1, Step 0, Loss: 2.417486, Accuracy: 6.25%
2026-10-03 14:21:36.625 | INFO     | train:train_epoch:47 - Epoch 1, Step 100, Loss: 1.296951, Accuracy: 55.29%
2026-10-03 14:21:36.979 | INFO     | train:train_epoch:47 - Epoch 1, Step 200, Loss: 1.060950, Accuracy: 63.22%
2026-10-03 14:21:37.232 | INFO     | train:test_epoch:90 - Epoch 1, Val Loss: 0.762914, Val Accuracy: 72.98%
2026-10-03 14:21:37.234 | INFO     | train:linear_probe:190 - New best vali

In [ ]:
!CUDA_VISIBLE_DEVICES=1 WANDB_MODE=disabled python generalization_ct.py \
    --model resnet50 \
    --pretrained_ds imagenet \
    --transfer_ds medmnist/dermamnist \
    --seed 43 \
    --linear_probe_train_bs 32 \
    --linear_probe_test_bs 128 \
    --epochs 20

2026-10-03 15:01:58.735 | INFO     | __main__:main:74 - Log file: ./logs/generalization_ct_imagenet_to_medmnist-dermamnist_resnet50_seed43_epochs20.log
2026-10-03 15:01:58.738 | INFO     | __main__:main:78 - Running on cuda
2026-10-03 15:02:01.413 | INFO     | __main__:main:105 - Testing baseline...
2026-10-03 15:02:01.486 | INFO     | __main__:main:116 - Number of trainable parameters: 14343
2026-10-03 15:02:01.486 | INFO     | __main__:main:117 - Starting transfer learning...
2026-10-03 15:02:31.397 | INFO     | train:train_epoch:47 - Epoch 1, Step 0, Loss: 1.977283, Accuracy: 3.12%
2026-10-03 15:02:31.788 | INFO     | train:train_epoch:47 - Epoch 1, Step 100, Loss: 1.216802, Accuracy: 60.33%
2026-10-03 15:02:32.189 | INFO     | train:train_epoch:47 - Epoch 1, Step 200, Loss: 1.022703, Accuracy: 66.06%
2026-10-03 15:02:32.480 | INFO     | train:test_epoch:90 - Epoch 1, Val Loss: 0.767136, Val Accuracy: 72.88%
2026-10-03 15:02:32.482 | INFO     | train:linear_probe:190 - New best vali

In [ ]:
!CUDA_VISIBLE_DEVICES=1 WANDB_MODE=disabled python generalization_ct.py \
    --model resnet50 \
    --pretrained_ds imagenet \
    --transfer_ds medmnist/dermamnist \
    --seed 44 \
    --linear_probe_train_bs 32 \
    --linear_probe_test_bs 128 \
    --epochs 20

2026-10-03 15:44:46.380 | INFO     | __main__:main:74 - Log file: ./logs/generalization_ct_imagenet_to_medmnist-dermamnist_resnet50_seed44_epochs20.log
2026-10-03 15:44:46.385 | INFO     | __main__:main:78 - Running on cuda
2026-10-03 15:44:49.267 | INFO     | __main__:main:105 - Testing baseline...
2026-10-03 15:44:49.349 | INFO     | __main__:main:116 - Number of trainable parameters: 14343
2026-10-03 15:44:49.349 | INFO     | __main__:main:117 - Starting transfer learning...
2026-10-03 15:45:21.255 | INFO     | train:train_epoch:47 - Epoch 1, Step 0, Loss: 2.038107, Accuracy: 9.38%
2026-10-03 15:45:21.694 | INFO     | train:train_epoch:47 - Epoch 1, Step 100, Loss: 1.198566, Accuracy: 59.56%
2026-10-03 15:45:22.064 | INFO     | train:train_epoch:47 - Epoch 1, Step 200, Loss: 1.006768, Accuracy: 65.53%
2026-10-03 15:45:22.451 | INFO     | train:test_epoch:90 - Epoch 1, Val Loss: 0.785562, Val Accuracy: 72.78%
2026-10-03 15:45:22.452 | INFO     | train:linear_probe:190 - New best vali

### Learned SW-CT with Fixed Beta Initialization

The original Learned SW-CT configuration is first reproduced using the fixed initialization:

\[[\beta_1,\beta_2,\beta_3,\beta_4]=[0.78,0.78,0.78,0.78].\]

The initial value is only a starting point. During training, the four beta parameters are optimized independently together with the downstream classifier.

These runs provide the reference against which random initialization will be compared.

In [ ]:
!CUDA_VISIBLE_DEVICES=1 WANDB_MODE=disabled python generalization_stagewise_ct.py \
    --model resnet50 \
    --pretrained_ds imagenet \
    --transfer_ds medmnist/dermamnist \
    --seed 42 \
    --transfer_train_bs 32 \
    --transfer_test_bs 128 \
    --init_strategy fixed \
    --init_beta 0.78 \
    --beta_lr 0.01 \
    --epochs 20

2026-10-03 16:28:02.028 | INFO     | __main__:main:286 - Log file: ./logs/generalization_stagewise_ct_imagenet_to_medmnist-dermamnist_resnet50_seed42_epochs20_initfixed0.78_betalr0.01.log
2026-10-03 16:28:02.108 | INFO     | __main__:main:302 - Running on cuda
2026-10-03 16:28:04.812 | INFO     | __main__:main:389 - Initialization strategy: fixed
2026-10-03 16:28:04.812 | INFO     | __main__:main:394 - Requested initial stage betas: [0.78, 0.78, 0.78, 0.78]
Stage-Wise CT ReLU counts: [13, 16, 24, 12]
2026-10-03 16:28:04.886 | INFO     | __main__:main:418 - Initial stage betas: [0.7799999713897705, 0.7799999713897705, 0.7799999713897705, 0.7799999713897705]
2026-10-03 16:28:04.887 | INFO     | __main__:main:434 - Trainable curvature parameters: 4
2026-10-03 16:28:04.887 | INFO     | __main__:main:439 - Total trainable parameters: 14347
2026-10-03 16:28:06.337 | INFO     | train:train_epoch:47 - Epoch 1, Step 0, Loss: 2.155895, Accuracy: 0.00%
2026-10-03 16:29:06.637 | INFO     | train:t

In [ ]:
!CUDA_VISIBLE_DEVICES=1 WANDB_MODE=disabled python generalization_stagewise_ct.py \
    --model resnet50 \
    --pretrained_ds imagenet \
    --transfer_ds medmnist/dermamnist \
    --seed 43 \
    --transfer_train_bs 32 \
    --transfer_test_bs 128 \
    --init_strategy fixed \
    --init_beta 0.78 \
    --beta_lr 0.01 \
    --epochs 20

2026-10-03 17:14:26.190 | INFO     | __main__:main:286 - Log file: ./logs/generalization_stagewise_ct_imagenet_to_medmnist-dermamnist_resnet50_seed43_epochs20_initfixed0.78_betalr0.01.log
2026-10-03 17:14:26.296 | INFO     | __main__:main:302 - Running on cuda
2026-10-03 17:14:29.088 | INFO     | __main__:main:389 - Initialization strategy: fixed
2026-10-03 17:14:29.088 | INFO     | __main__:main:394 - Requested initial stage betas: [0.78, 0.78, 0.78, 0.78]
Stage-Wise CT ReLU counts: [13, 16, 24, 12]
2026-10-03 17:14:29.188 | INFO     | __main__:main:418 - Initial stage betas: [0.7799999713897705, 0.7799999713897705, 0.7799999713897705, 0.7799999713897705]
2026-10-03 17:14:29.190 | INFO     | __main__:main:434 - Trainable curvature parameters: 4
2026-10-03 17:14:29.190 | INFO     | __main__:main:439 - Total trainable parameters: 14347
2026-10-03 17:14:30.743 | INFO     | train:train_epoch:47 - Epoch 1, Step 0, Loss: 2.044997, Accuracy: 9.38%
2026-10-03 17:15:30.947 | INFO     | train:t

In [ ]:
!CUDA_VISIBLE_DEVICES=1 WANDB_MODE=disabled python generalization_stagewise_ct.py \
    --model resnet50 \
    --pretrained_ds imagenet \
    --transfer_ds medmnist/dermamnist \
    --seed 44 \
    --transfer_train_bs 32 \
    --transfer_test_bs 128 \
    --init_strategy fixed \
    --init_beta 0.78 \
    --beta_lr 0.01 \
    --epochs 20

2026-10-03 18:00:49.535 | INFO     | __main__:main:286 - Log file: ./logs/generalization_stagewise_ct_imagenet_to_medmnist-dermamnist_resnet50_seed44_epochs20_initfixed0.78_betalr0.01.log
2026-10-03 18:00:49.639 | INFO     | __main__:main:302 - Running on cuda
2026-10-03 18:00:52.465 | INFO     | __main__:main:389 - Initialization strategy: fixed
2026-10-03 18:00:52.465 | INFO     | __main__:main:394 - Requested initial stage betas: [0.78, 0.78, 0.78, 0.78]
Stage-Wise CT ReLU counts: [13, 16, 24, 12]
2026-10-03 18:00:52.555 | INFO     | __main__:main:418 - Initial stage betas: [0.7799999713897705, 0.7799999713897705, 0.7799999713897705, 0.7799999713897705]
2026-10-03 18:00:52.557 | INFO     | __main__:main:434 - Trainable curvature parameters: 4
2026-10-03 18:00:52.557 | INFO     | __main__:main:439 - Total trainable parameters: 14347
2026-10-03 18:00:54.123 | INFO     | train:train_epoch:47 - Epoch 1, Step 0, Loss: 1.860342, Accuracy: 40.62%
2026-10-03 18:01:54.371 | INFO     | train:

### Learned SW-CT with Random Beta Initialization

The fixed beta = 0.78 initialization may raise the question of whether Learned SW-CT performs well only because its starting point was informed by the earlier S-CT experiments.

To test this dependency, each of the four stage-wise beta parameters is now initialized independently from a uniform distribution between 0.70 and 0.99.

The initialization is reproducible: the random generator is seeded using the experiment seed. Therefore, each seed always receives the same initial beta vector.

If random initialization produces similar performance to the fixed 0.78 experiment, this provides evidence that the Learned SW-CT result is not dependent on a single hand-selected initialization.

In [ ]:
!CUDA_VISIBLE_DEVICES=1 WANDB_MODE=disabled python generalization_stagewise_ct.py \
    --model resnet50 \
    --pretrained_ds imagenet \
    --transfer_ds medmnist/dermamnist \
    --seed 42 \
    --transfer_train_bs 32 \
    --transfer_test_bs 128 \
    --init_strategy random \
    --beta_lr 0.01 \
    --epochs 20

2026-10-03 18:47:11.393 | INFO     | __main__:main:286 - Log file: ./logs/generalization_stagewise_ct_imagenet_to_medmnist-dermamnist_resnet50_seed42_epochs20_initrandom_betalr0.01.log
2026-10-03 18:47:11.507 | INFO     | __main__:main:302 - Running on cuda
2026-10-03 18:47:14.290 | INFO     | __main__:main:389 - Initialization strategy: random
2026-10-03 18:47:14.291 | INFO     | __main__:main:394 - Requested initial stage betas: [0.924447, 0.827275, 0.948993, 0.902237]
Stage-Wise CT ReLU counts: [13, 16, 24, 12]
2026-10-03 18:47:14.421 | INFO     | __main__:main:418 - Initial stage betas: [0.924447238445282, 0.827274739742279, 0.9489933848381042, 0.9022367000579834]
2026-10-03 18:47:14.423 | INFO     | __main__:main:434 - Trainable curvature parameters: 4
2026-10-03 18:47:14.424 | INFO     | __main__:main:439 - Total trainable parameters: 14347
2026-10-03 18:47:15.902 | INFO     | train:train_epoch:47 - Epoch 1, Step 0, Loss: 2.149691, Accuracy: 0.00%
2026-10-03 18:48:16.269 | INFO  

In [ ]:
!CUDA_VISIBLE_DEVICES=1 WANDB_MODE=disabled python generalization_stagewise_ct.py \
    --model resnet50 \
    --pretrained_ds imagenet \
    --transfer_ds medmnist/dermamnist \
    --seed 43 \
    --transfer_train_bs 32 \
    --transfer_test_bs 128 \
    --init_strategy random \
    --beta_lr 0.01 \
    --epochs 20

2026-10-03 19:33:34.928 | INFO     | __main__:main:286 - Log file: ./logs/generalization_stagewise_ct_imagenet_to_medmnist-dermamnist_resnet50_seed43_epochs20_initrandom_betalr0.01.log
2026-10-03 19:33:34.991 | INFO     | __main__:main:302 - Running on cuda
2026-10-03 19:33:37.755 | INFO     | __main__:main:389 - Initialization strategy: random
2026-10-03 19:33:37.756 | INFO     | __main__:main:394 - Requested initial stage betas: [0.889167, 0.712695, 0.705809, 0.943372]
Stage-Wise CT ReLU counts: [13, 16, 24, 12]
2026-10-03 19:33:37.849 | INFO     | __main__:main:418 - Initial stage betas: [0.8891667127609253, 0.7126947641372681, 0.7058085799217224, 0.9433716535568237]
2026-10-03 19:33:37.851 | INFO     | __main__:main:434 - Trainable curvature parameters: 4
2026-10-03 19:33:37.851 | INFO     | __main__:main:439 - Total trainable parameters: 14347
2026-10-03 19:33:39.366 | INFO     | train:train_epoch:47 - Epoch 1, Step 0, Loss: 2.040869, Accuracy: 12.50%
2026-10-03 19:34:38.999 | INF

In [3]:
!CUDA_VISIBLE_DEVICES=1 WANDB_MODE=disabled python generalization_stagewise_ct.py \
    --model resnet50 \
    --pretrained_ds imagenet \
    --transfer_ds medmnist/dermamnist \
    --seed 44 \
    --transfer_train_bs 32 \
    --transfer_test_bs 128 \
    --init_strategy random \
    --beta_lr 0.01 \
    --epochs 20

2026-10-04 11:55:38.973 | INFO     | __main__:main:286 - Log file: ./logs/generalization_stagewise_ct_imagenet_to_medmnist-dermamnist_resnet50_seed44_epochs20_initrandom_betalr0.01.log
2026-10-04 11:55:39.037 | INFO     | __main__:main:302 - Running on cuda
Downloading: "https://download.pytorch.org/models/resnet50-0676ba61.pth" to /root/.cache/torch/hub/checkpoints/resnet50-0676ba61.pth
100%|███████████████████████████████████████| 97.8M/97.8M [00:00<00:00, 172MB/s]
2026-10-04 11:55:42.616 | INFO     | __main__:main:389 - Initialization strategy: random
2026-10-04 11:55:42.616 | INFO     | __main__:main:394 - Requested initial stage betas: [0.735544, 0.774853, 0.817674, 0.981063]
Stage-Wise CT ReLU counts: [13, 16, 24, 12]
2026-10-04 11:55:42.714 | INFO     | __main__:main:418 - Initial stage betas: [0.7355440258979797, 0.7748528122901917, 0.8176735043525696, 0.9810633659362793]
2026-10-04 11:55:42.715 | INFO     | __main__:main:434 - Trainable curvature parameters: 4
2026-10-04 11:55

# ResNet-50 — 30-Epoch Evaluation

The same experiments are now repeated for 30 epochs.

The purpose of increasing the training budget is to determine whether the 20-epoch results are limited by insufficient optimization.

If a method improves substantially with 30 epochs, this suggests that additional optimization is useful. If performance remains similar, it indicates that the method has largely stabilized within the original 20-epoch training budget.

All other experimental settings remain unchanged.

For random-initialization SW-CT, the same seed produces exactly the same initial four-beta vector in both the 20-epoch and 30-epoch experiments. Therefore, initialization is held constant while the training duration changes.

Execution time is again recorded to measure the computational cost of extending training from 20 to 30 epochs.

In [4]:
!CUDA_VISIBLE_DEVICES=1 WANDB_MODE=disabled python generalization_ct.py \
    --model resnet50 \
    --pretrained_ds imagenet \
    --transfer_ds medmnist/dermamnist \
    --seed 42 \
    --linear_probe_train_bs 32 \
    --linear_probe_test_bs 128 \
    --epochs 30

2026-10-04 12:39:58.548 | INFO     | __main__:main:74 - Log file: ./logs/generalization_ct_imagenet_to_medmnist-dermamnist_resnet50_seed42_epochs30.log
2026-10-04 12:39:58.550 | INFO     | __main__:main:78 - Running on cuda
2026-10-04 12:40:00.827 | INFO     | __main__:main:105 - Testing baseline...
2026-10-04 12:40:00.892 | INFO     | __main__:main:116 - Number of trainable parameters: 14343
2026-10-04 12:40:00.892 | INFO     | __main__:main:117 - Starting transfer learning...
2026-10-04 12:40:27.873 | INFO     | train:train_epoch:47 - Epoch 1, Step 0, Loss: 2.417486, Accuracy: 6.25%
2026-10-04 12:40:28.219 | INFO     | train:train_epoch:47 - Epoch 1, Step 100, Loss: 1.296951, Accuracy: 55.29%
2026-10-04 12:40:28.575 | INFO     | train:train_epoch:47 - Epoch 1, Step 200, Loss: 1.060950, Accuracy: 63.22%
2026-10-04 12:40:28.818 | INFO     | train:test_epoch:90 - Epoch 1, Val Loss: 0.762914, Val Accuracy: 72.98%
2026-10-04 12:40:28.820 | INFO     | train:linear_probe:190 - New best vali

In [5]:
!CUDA_VISIBLE_DEVICES=1 WANDB_MODE=disabled python generalization_ct.py \
    --model resnet50 \
    --pretrained_ds imagenet \
    --transfer_ds medmnist/dermamnist \
    --seed 43 \
    --linear_probe_train_bs 32 \
    --linear_probe_test_bs 128 \
    --epochs 30

2026-10-04 13:23:03.228 | INFO     | __main__:main:74 - Log file: ./logs/generalization_ct_imagenet_to_medmnist-dermamnist_resnet50_seed43_epochs30.log
2026-10-04 13:23:03.232 | INFO     | __main__:main:78 - Running on cuda
2026-10-04 13:23:05.776 | INFO     | __main__:main:105 - Testing baseline...
2026-10-04 13:23:05.827 | INFO     | __main__:main:116 - Number of trainable parameters: 14343
2026-10-04 13:23:05.828 | INFO     | __main__:main:117 - Starting transfer learning...
2026-10-04 13:23:31.738 | INFO     | train:train_epoch:47 - Epoch 1, Step 0, Loss: 1.977283, Accuracy: 3.12%
2026-10-04 13:23:32.081 | INFO     | train:train_epoch:47 - Epoch 1, Step 100, Loss: 1.216802, Accuracy: 60.33%
2026-10-04 13:23:32.427 | INFO     | train:train_epoch:47 - Epoch 1, Step 200, Loss: 1.022703, Accuracy: 66.06%
2026-10-04 13:23:32.676 | INFO     | train:test_epoch:90 - Epoch 1, Val Loss: 0.767136, Val Accuracy: 72.88%
2026-10-04 13:23:32.677 | INFO     | train:linear_probe:190 - New best vali

In [6]:
!CUDA_VISIBLE_DEVICES=1 WANDB_MODE=disabled python generalization_ct.py \
    --model resnet50 \
    --pretrained_ds imagenet \
    --transfer_ds medmnist/dermamnist \
    --seed 44 \
    --linear_probe_train_bs 32 \
    --linear_probe_test_bs 128 \
    --epochs 30

2026-10-04 14:06:12.325 | INFO     | __main__:main:74 - Log file: ./logs/generalization_ct_imagenet_to_medmnist-dermamnist_resnet50_seed44_epochs30.log
2026-10-04 14:06:12.327 | INFO     | __main__:main:78 - Running on cuda
2026-10-04 14:06:14.820 | INFO     | __main__:main:105 - Testing baseline...
2026-10-04 14:06:14.867 | INFO     | __main__:main:116 - Number of trainable parameters: 14343
2026-10-04 14:06:14.868 | INFO     | __main__:main:117 - Starting transfer learning...
2026-10-04 14:06:40.804 | INFO     | train:train_epoch:47 - Epoch 1, Step 0, Loss: 2.038107, Accuracy: 9.38%
2026-10-04 14:06:41.143 | INFO     | train:train_epoch:47 - Epoch 1, Step 100, Loss: 1.198566, Accuracy: 59.56%
2026-10-04 14:06:41.501 | INFO     | train:train_epoch:47 - Epoch 1, Step 200, Loss: 1.006768, Accuracy: 65.53%
2026-10-04 14:06:41.764 | INFO     | train:test_epoch:90 - Epoch 1, Val Loss: 0.785562, Val Accuracy: 72.78%
2026-10-04 14:06:41.765 | INFO     | train:linear_probe:190 - New best vali

In [7]:
!CUDA_VISIBLE_DEVICES=1 WANDB_MODE=disabled python generalization_stagewise_ct.py \
    --model resnet50 \
    --pretrained_ds imagenet \
    --transfer_ds medmnist/dermamnist \
    --seed 42 \
    --transfer_train_bs 32 \
    --transfer_test_bs 128 \
    --init_strategy fixed \
    --init_beta 0.78 \
    --beta_lr 0.01 \
    --epochs 30

2026-10-04 14:49:33.842 | INFO     | __main__:main:286 - Log file: ./logs/generalization_stagewise_ct_imagenet_to_medmnist-dermamnist_resnet50_seed42_epochs30_initfixed0.78_betalr0.01.log
2026-10-04 14:49:33.912 | INFO     | __main__:main:302 - Running on cuda
2026-10-04 14:49:36.257 | INFO     | __main__:main:389 - Initialization strategy: fixed
2026-10-04 14:49:36.257 | INFO     | __main__:main:394 - Requested initial stage betas: [0.78, 0.78, 0.78, 0.78]
Stage-Wise CT ReLU counts: [13, 16, 24, 12]
2026-10-04 14:49:36.327 | INFO     | __main__:main:418 - Initial stage betas: [0.7799999713897705, 0.7799999713897705, 0.7799999713897705, 0.7799999713897705]
2026-10-04 14:49:36.329 | INFO     | __main__:main:434 - Trainable curvature parameters: 4
2026-10-04 14:49:36.329 | INFO     | __main__:main:439 - Total trainable parameters: 14347
2026-10-04 14:49:37.746 | INFO     | train:train_epoch:47 - Epoch 1, Step 0, Loss: 2.155895, Accuracy: 0.00%
2026-10-04 14:50:33.327 | INFO     | train:t

In [8]:
!CUDA_VISIBLE_DEVICES=1 WANDB_MODE=disabled python generalization_stagewise_ct.py \
    --model resnet50 \
    --pretrained_ds imagenet \
    --transfer_ds medmnist/dermamnist \
    --seed 43 \
    --transfer_train_bs 32 \
    --transfer_test_bs 128 \
    --init_strategy fixed \
    --init_beta 0.78 \
    --beta_lr 0.01 \
    --epochs 30

2026-10-04 15:55:53.810 | INFO     | __main__:main:286 - Log file: ./logs/generalization_stagewise_ct_imagenet_to_medmnist-dermamnist_resnet50_seed43_epochs30_initfixed0.78_betalr0.01.log
2026-10-04 15:55:53.896 | INFO     | __main__:main:302 - Running on cuda
2026-10-04 15:55:56.599 | INFO     | __main__:main:389 - Initialization strategy: fixed
2026-10-04 15:55:56.599 | INFO     | __main__:main:394 - Requested initial stage betas: [0.78, 0.78, 0.78, 0.78]
Stage-Wise CT ReLU counts: [13, 16, 24, 12]
2026-10-04 15:55:56.684 | INFO     | __main__:main:418 - Initial stage betas: [0.7799999713897705, 0.7799999713897705, 0.7799999713897705, 0.7799999713897705]
2026-10-04 15:55:56.685 | INFO     | __main__:main:434 - Trainable curvature parameters: 4
2026-10-04 15:55:56.685 | INFO     | __main__:main:439 - Total trainable parameters: 14347
2026-10-04 15:55:58.145 | INFO     | train:train_epoch:47 - Epoch 1, Step 0, Loss: 2.044997, Accuracy: 9.38%
2026-10-04 15:56:54.533 | INFO     | train:t

In [9]:
!CUDA_VISIBLE_DEVICES=1 WANDB_MODE=disabled python generalization_stagewise_ct.py \
    --model resnet50 \
    --pretrained_ds imagenet \
    --transfer_ds medmnist/dermamnist \
    --seed 44 \
    --transfer_train_bs 32 \
    --transfer_test_bs 128 \
    --init_strategy fixed \
    --init_beta 0.78 \
    --beta_lr 0.01 \
    --epochs 30

2026-10-04 17:02:16.875 | INFO     | __main__:main:286 - Log file: ./logs/generalization_stagewise_ct_imagenet_to_medmnist-dermamnist_resnet50_seed44_epochs30_initfixed0.78_betalr0.01.log
2026-10-04 17:02:16.937 | INFO     | __main__:main:302 - Running on cuda
2026-10-04 17:02:19.274 | INFO     | __main__:main:389 - Initialization strategy: fixed
2026-10-04 17:02:19.275 | INFO     | __main__:main:394 - Requested initial stage betas: [0.78, 0.78, 0.78, 0.78]
Stage-Wise CT ReLU counts: [13, 16, 24, 12]
2026-10-04 17:02:19.376 | INFO     | __main__:main:418 - Initial stage betas: [0.7799999713897705, 0.7799999713897705, 0.7799999713897705, 0.7799999713897705]
2026-10-04 17:02:19.377 | INFO     | __main__:main:434 - Trainable curvature parameters: 4
2026-10-04 17:02:19.377 | INFO     | __main__:main:439 - Total trainable parameters: 14347
2026-10-04 17:02:20.824 | INFO     | train:train_epoch:47 - Epoch 1, Step 0, Loss: 1.860342, Accuracy: 40.62%
2026-10-04 17:03:17.220 | INFO     | train:

In [10]:
!CUDA_VISIBLE_DEVICES=1 WANDB_MODE=disabled python generalization_stagewise_ct.py \
    --model resnet50 \
    --pretrained_ds imagenet \
    --transfer_ds medmnist/dermamnist \
    --seed 42 \
    --transfer_train_bs 32 \
    --transfer_test_bs 128 \
    --init_strategy random \
    --beta_lr 0.01 \
    --epochs 30

2026-10-04 18:08:39.033 | INFO     | __main__:main:286 - Log file: ./logs/generalization_stagewise_ct_imagenet_to_medmnist-dermamnist_resnet50_seed42_epochs30_initrandom_betalr0.01.log
2026-10-04 18:08:39.096 | INFO     | __main__:main:302 - Running on cuda
2026-10-04 18:08:41.400 | INFO     | __main__:main:389 - Initialization strategy: random
2026-10-04 18:08:41.401 | INFO     | __main__:main:394 - Requested initial stage betas: [0.924447, 0.827275, 0.948993, 0.902237]
Stage-Wise CT ReLU counts: [13, 16, 24, 12]
2026-10-04 18:08:41.469 | INFO     | __main__:main:418 - Initial stage betas: [0.924447238445282, 0.827274739742279, 0.9489933848381042, 0.9022367000579834]
2026-10-04 18:08:41.470 | INFO     | __main__:main:434 - Trainable curvature parameters: 4
2026-10-04 18:08:41.470 | INFO     | __main__:main:439 - Total trainable parameters: 14347
2026-10-04 18:08:42.861 | INFO     | train:train_epoch:47 - Epoch 1, Step 0, Loss: 2.149691, Accuracy: 0.00%
2026-10-04 18:09:39.345 | INFO  

In [11]:
!CUDA_VISIBLE_DEVICES=1 WANDB_MODE=disabled python generalization_stagewise_ct.py \
    --model resnet50 \
    --pretrained_ds imagenet \
    --transfer_ds medmnist/dermamnist \
    --seed 43 \
    --transfer_train_bs 32 \
    --transfer_test_bs 128 \
    --init_strategy random \
    --beta_lr 0.01 \
    --epochs 30

2026-10-04 19:15:00.825 | INFO     | __main__:main:286 - Log file: ./logs/generalization_stagewise_ct_imagenet_to_medmnist-dermamnist_resnet50_seed43_epochs30_initrandom_betalr0.01.log
2026-10-04 19:15:00.898 | INFO     | __main__:main:302 - Running on cuda
2026-10-04 19:15:03.187 | INFO     | __main__:main:389 - Initialization strategy: random
2026-10-04 19:15:03.188 | INFO     | __main__:main:394 - Requested initial stage betas: [0.889167, 0.712695, 0.705809, 0.943372]
Stage-Wise CT ReLU counts: [13, 16, 24, 12]
2026-10-04 19:15:03.278 | INFO     | __main__:main:418 - Initial stage betas: [0.8891667127609253, 0.7126947641372681, 0.7058085799217224, 0.9433716535568237]
2026-10-04 19:15:03.279 | INFO     | __main__:main:434 - Trainable curvature parameters: 4
2026-10-04 19:15:03.280 | INFO     | __main__:main:439 - Total trainable parameters: 14347
2026-10-04 19:15:04.678 | INFO     | train:train_epoch:47 - Epoch 1, Step 0, Loss: 2.040869, Accuracy: 12.50%
2026-10-04 19:16:01.197 | INF

In [12]:
!CUDA_VISIBLE_DEVICES=1 WANDB_MODE=disabled python generalization_stagewise_ct.py \
    --model resnet50 \
    --pretrained_ds imagenet \
    --transfer_ds medmnist/dermamnist \
    --seed 44 \
    --transfer_train_bs 32 \
    --transfer_test_bs 128 \
    --init_strategy random \
    --beta_lr 0.01 \
    --epochs 30

2026-10-04 20:21:19.906 | INFO     | __main__:main:286 - Log file: ./logs/generalization_stagewise_ct_imagenet_to_medmnist-dermamnist_resnet50_seed44_epochs30_initrandom_betalr0.01.log
2026-10-04 20:21:20.019 | INFO     | __main__:main:302 - Running on cuda
2026-10-04 20:21:22.718 | INFO     | __main__:main:389 - Initialization strategy: random
2026-10-04 20:21:22.719 | INFO     | __main__:main:394 - Requested initial stage betas: [0.735544, 0.774853, 0.817674, 0.981063]
Stage-Wise CT ReLU counts: [13, 16, 24, 12]
2026-10-04 20:21:22.830 | INFO     | __main__:main:418 - Initial stage betas: [0.7355440258979797, 0.7748528122901917, 0.8176735043525696, 0.9810633659362793]
2026-10-04 20:21:22.832 | INFO     | __main__:main:434 - Trainable curvature parameters: 4
2026-10-04 20:21:22.833 | INFO     | __main__:main:439 - Total trainable parameters: 14347
2026-10-04 20:21:24.444 | INFO     | train:train_epoch:47 - Epoch 1, Step 0, Loss: 1.866424, Accuracy: 43.75%
2026-10-04 20:22:21.037 | INF